# MedIA — Notebook 06 | Camada Gold (versão integral revisada)

**Microsoft Fabric · PySpark · Delta Lake · Modelagem dimensional**

Este notebook recompõe a lógica fornecida pelo projeto: dimensões CID/tempo, fatos SIH/SIM, fatos CNES condicionais, controle de cobertura, indicadores por UF/ano, validação e auditoria.

### Alteração essencial para o Power BI
`codigo_uf_ibge` será armazenado como **inteiro** nas fatos, alinhado a `gold_dim_uf[uf_codigo]`. O código municipal de seis dígitos continua **texto** para preservar seu identificador. A versão anual do arquivo SIM não impede análises mensais **por data de óbito**, pois essa data é conhecida na Silver; a cobertura do arquivo permanece identificada como anual.

### Segurança antes de executar
1. Anexe **`lh_media_analytics` (origem)** como Lakehouse padrão, **não** `lh_media_bi` (atalhos).
2. Execute com `APLICAR_GRAVACAO = False` para fazer as validações **sem sobrescrever** as Gold.
3. Só depois de conferir os resultados e realizar backup/confirmar o ambiente altere para `True` e reexecute.
4. A reconstrução afeta tabelas já utilizadas no modelo semântico; sincronize o esquema e valide o Direct Lake após a gravação.
5. Os códigos de UF na Silver devem ser códigos municipais compatíveis. O notebook falha em dados inconsistentes em vez de fazer associação incorreta.

**Limitação:** a reconstrução de múltiplas tabelas Gold não é uma transação atômica. Uma falha durante a gravação pode deixar algumas tabelas atualizadas e outras não.

## 01 — Configuração e pré-requisitos

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql import types as T

NOTEBOOK = "nb_media_06_gold"
EXECUTION_ID = str(uuid4())
INICIO = datetime.now(timezone.utc).replace(tzinfo=None)
SCHEMA = "dbo"

# PRIMEIRO EXECUTE SEM GRAVAR. Depois de validar, troque para True.
APLICAR_GRAVACAO = False
CONFIRMO_LAKEHOUSE_ORIGINAL = False  # True somente após conferir lh_media_analytics

# As taxas populacionais exigem denominadores oficiais validados.
CALCULAR_TAXAS_POPULACIONAIS = False

print("=" * 70)
print("MedIA | Notebook 06 — GOLD")
print("Execução:", EXECUTION_ID)
print("Spark:", spark.version)
print("Gravação habilitada:", APLICAR_GRAVACAO)
print("=" * 70)

if CALCULAR_TAXAS_POPULACIONAIS:
    raise RuntimeError("Taxas populacionais requerem população IBGE validada.")

# A confirmação explícita impede regravação acidental no Lakehouse de BI.
if APLICAR_GRAVACAO and not CONFIRMO_LAKEHOUSE_ORIGINAL:
    raise RuntimeError(
        "Confirme o Lakehouse original lh_media_analytics antes de gravar: "
        "defina CONFIRMO_LAKEHOUSE_ORIGINAL=True."
    )

OBRIGATORIAS = ["dbo.silver_sih_internacoes", "dbo.silver_sim_obitos"]
OPCIONAIS = [
    "dbo.silver_cnes_estabelecimentos", "dbo.silver_cnes_leitos",
    "dbo.gold_dim_regiao", "dbo.gold_dim_uf", "dbo.gold_dim_localidade"
]

for tabela in OBRIGATORIAS:
    if not spark.catalog.tableExists(tabela):
        raise RuntimeError(f"Tabela ausente: {tabela}. Execute nb_media_05_silver.")
    print("OK:", tabela)
for tabela in OPCIONAIS:
    print("DISPONÍVEL" if spark.catalog.tableExists(tabela) else "AUSENTE", tabela)

print("\nAtenção: Spark gravará no Lakehouse padrão do notebook.")

## 02 — Funções de apoio, chaves e geografia

In [ ]:
MAPA_UF = {
    "11":"RO", "12":"AC", "13":"AM", "14":"RR", "15":"PA", "16":"AP",
    "17":"TO", "21":"MA", "22":"PI", "23":"CE", "24":"RN", "25":"PB",
    "26":"PE", "27":"AL", "28":"SE", "29":"BA", "31":"MG", "32":"ES",
    "33":"RJ", "35":"SP", "41":"PR", "42":"SC", "43":"RS", "50":"MS",
    "51":"MT", "52":"GO", "53":"DF"
}

UF_NOME = {
    "AC":"Acre", "AL":"Alagoas", "AP":"Amapá", "AM":"Amazonas",
    "BA":"Bahia", "CE":"Ceará", "DF":"Distrito Federal",
    "ES":"Espírito Santo", "GO":"Goiás", "MA":"Maranhão",
    "MT":"Mato Grosso", "MS":"Mato Grosso do Sul", "MG":"Minas Gerais",
    "PA":"Pará", "PB":"Paraíba", "PR":"Paraná", "PE":"Pernambuco",
    "PI":"Piauí", "RJ":"Rio de Janeiro", "RN":"Rio Grande do Norte",
    "RS":"Rio Grande do Sul", "RO":"Rondônia", "RR":"Roraima",
    "SC":"Santa Catarina", "SP":"São Paulo", "SE":"Sergipe",
    "TO":"Tocantins"
}

MAPA_UF_EXPR = F.create_map(
    *[elemento for codigo, uf in MAPA_UF.items()
      for elemento in (F.lit(codigo), F.lit(uf))]
)
UF_NOME_EXPR = F.create_map(
    *[elemento for uf, nome in UF_NOME.items()
      for elemento in (F.lit(uf), F.lit(nome))]
)


def verificar_colunas(df, obrigatorias, nome):
    faltantes = sorted(set(obrigatorias) - set(df.columns))
    if faltantes:
        raise ValueError(f"{nome}: faltam colunas {faltantes}")


def codigo_municipio_6(coluna):
    """Normaliza identificadores municipais para o padrão DATASUS de 6 dígitos.
    Para código IBGE de 7 dígitos, usa os 6 primeiros SOMENTE para
    compatibilização, que ainda precisa ser validada em nível municipal.
    """
    codigo = F.trim(coluna.cast("string"))
    return (F.when(codigo.rlike(r"^[0-9]{6}$"), codigo)
             .when(codigo.rlike(r"^[0-9]{7}$"), F.substring(codigo, 1, 6))
             .otherwise(F.lit(None).cast("string")))


def adicionar_geografia(df, coluna_municipio):
    df = df.withColumn("municipio_codigo_6", codigo_municipio_6(F.col(coluna_municipio)))
    # CORREÇÃO: codigo_uf_ibge agora é INTEGER, não STRING.
    df = df.withColumn(
        "codigo_uf_ibge",
        F.substring(F.col("municipio_codigo_6"), 1, 2).cast(T.IntegerType())
    )
    df = df.withColumn(
        "uf_residencia",
        F.element_at(MAPA_UF_EXPR, F.col("codigo_uf_ibge").cast("string"))
    )
    return df


def cid_padronizado(coluna):
    return F.upper(F.trim(coluna.cast("string")))


def validar_geografia(df, nome, coluna_peso=None):
    invalidos = df.filter(
        F.col("codigo_uf_ibge").isNull() |
        F.col("uf_residencia").isNull()
    )
    qtd = invalidos.count()
    if qtd:
        print(f"ERRO geográfico em {nome}: {qtd} linhas/grupos sem UF válida")
        display(invalidos.limit(10))
        raise RuntimeError(f"{nome}: existem códigos municipais/UF não reconhecidos.")
    print(f"Geografia OK — {nome}.")


def validar_chave_uf_dimensao():
    """A dimensão preexistente precisa ter chave única e domínio correto."""
    tabela = "dbo.gold_dim_uf"
    if not spark.catalog.tableExists(tabela):
        print("ATENÇÃO: gold_dim_uf ausente; valide geografia antes de criar relacionamentos.")
        return None
    df = spark.table(tabela)
    verificar_colunas(df, ["uf_codigo"], tabela)
    invalidas = df.filter(
        F.col("uf_codigo").isNull() |
        ~F.col("uf_codigo").cast("string").rlike(r"^[0-9]{2}$")
    ).count()
    duplicadas = df.groupBy("uf_codigo").count().filter("count > 1").count()
    if invalidas or duplicadas:
        raise RuntimeError(
            f"gold_dim_uf: {invalidas} chaves inválidas/nulas e "
            f"{duplicadas} chaves repetidas. Corrija na dimensão original."
        )
    return df.select(F.col("uf_codigo").cast("int").alias("uf_codigo")).distinct()


def validar_fato_contra_dim_uf(df, nome, dimensao):
    if dimensao is None:
        return
    faltantes = (df.select("codigo_uf_ibge").distinct()
                 .join(dimensao, F.col("codigo_uf_ibge") == F.col("uf_codigo"), "left_anti"))
    if faltantes.limit(1).count():
        display(faltantes)
        raise RuntimeError(f"{nome}: há UFs sem correspondência em gold_dim_uf.")


def validar_integralidade(nome, fato, coluna_medida, total_silver):
    total_gold = fato.agg(F.sum(coluna_medida).alias("total")).first()["total"]
    if int(total_gold or 0) != int(total_silver):
        raise RuntimeError(
            f"{nome}: Silver={total_silver}, Gold={total_gold}. "
            "Interrompido antes da gravação."
        )
    print(f"Reconciliação OK — {nome}: {int(total_gold):,} registros")


print("Funções de preparação e validação carregadas.")

## 03 — Ler e validar Silver SIH/SIM

In [ ]:
sih = spark.table("dbo.silver_sih_internacoes")
sim = spark.table("dbo.silver_sim_obitos")

verificar_colunas(sih, [
    "municipio_residencia", "cid_principal", "ano_competencia",
    "mes_competencia", "uf_arquivo", "ano_arquivo", "mes_arquivo",
    "valor_total", "valor_uti", "dias_permanencia", "indicador_obito",
    "cnes_estabelecimento"
], "SIH")

verificar_colunas(sim, [
    "municipio_residencia", "cid_causa_basica", "data_obito",
    "ano_obito", "uf_arquivo", "ano_arquivo", "sexo_codigo"
], "SIM")

TOTAL_SIH = sih.count()
TOTAL_SIM = sim.count()
if TOTAL_SIH <= 0 or TOTAL_SIM <= 0:
    raise RuntimeError("SIH ou SIM não possuem dados Silver.")

print("SIH Silver:", TOTAL_SIH)
print("SIM Silver:", TOTAL_SIM)
display(sim.groupBy("uf_arquivo", "ano_arquivo").count().orderBy("uf_arquivo", "ano_arquivo"))

# O arquivo nacional DO24OPEN não pode ser rotulado como uma carga estadual.
sim_geografia_teste = adicionar_geografia(sim, "municipio_residencia")
legado_suspeito = (
    sim_geografia_teste.filter(
        (F.col("uf_arquivo") != "BR") &
        F.col("uf_residencia").isNotNull() &
        (F.col("uf_residencia") != F.col("uf_arquivo"))
    ).limit(1).count()
)
if legado_suspeito:
    raise RuntimeError(
        "SIM tem arquivo marcado com UF estadual, mas municípios de residência de outra UF. "
        "Confira no nb_media_05_silver se DO24OPEN está corretamente registrado como BR."
    )

DIM_UF_PARA_VALIDACAO = validar_chave_uf_dimensao()
print("Pré-requisitos Silver OK.")

## 04 — Dimensão CID-10

In [ ]:
cid_sih = sih.select(cid_padronizado(F.col("cid_principal")).alias("cid_codigo"))
cid_sim = sim.select(cid_padronizado(F.col("cid_causa_basica")).alias("cid_codigo"))

gold_dim_cid = (
    cid_sih.unionByName(cid_sim)
    .filter(F.col("cid_codigo").isNotNull() & (F.col("cid_codigo") != ""))
    .distinct()
    .withColumn("cid_categoria", F.substring("cid_codigo", 1, 3))
    .withColumn("cid_capitulo_inicial", F.substring("cid_codigo", 1, 1))
    .withColumn("doenca_cardiovascular", F.col("cid_codigo").rlike(r"^I[0-9]{2}"))
    .withColumn("classificacao", F.lit("CID-10"))
)

assert gold_dim_cid.groupBy("cid_codigo").count().filter("count > 1").count() == 0
print("Dimensão CID:", gold_dim_cid.count())
display(gold_dim_cid.limit(10))

## 05 — Dimensão tempo (ano/mês de evento ou competência)

In [ ]:
periodos_sih = sih.select(
    F.col("ano_competencia").cast("int").alias("ano"),
    F.col("mes_competencia").cast("int").alias("mes")
)

# SIM é um arquivo ANUAL de origem, mas cada linha possui data_obito;
# usamos ano/mês da ocorrência do óbito — não da competência de arquivo.
periodos_sim = sim.select(
    F.year("data_obito").alias("ano"),
    F.month("data_obito").alias("mes")
)

periodos = (
    periodos_sih.unionByName(periodos_sim)
    .filter(F.col("ano").isNotNull() & F.col("mes").between(1, 12))
    .distinct()
)

gold_dim_tempo = (
    periodos.withColumn("ano_mes", F.col("ano") * 100 + F.col("mes"))
    .withColumn("data_inicio_mes", F.make_date(F.col("ano"), F.col("mes"), F.lit(1)))
    .withColumn("trimestre", F.quarter("data_inicio_mes"))
    .withColumn("semestre", F.when(F.col("mes") <= 6, 1).otherwise(2))
)

assert gold_dim_tempo.groupBy("ano_mes").count().filter("count > 1").count() == 0
print("Períodos na dimensão:", gold_dim_tempo.count())
display(gold_dim_tempo.orderBy("ano_mes"))

## 06 — Fato SIH: internações agregadas

In [ ]:
sih_geo = adicionar_geografia(sih, "municipio_residencia")
sih_geo = (
    sih_geo
    .withColumn("ano", F.col("ano_competencia").cast("int"))
    .withColumn("mes", F.col("mes_competencia").cast("int"))
    .withColumn("cid_codigo", cid_padronizado(F.col("cid_principal")))
    .withColumn(
        "obito_hospitalar",
        F.when(F.trim(F.col("indicador_obito").cast("string")) == "1", 1)
         .when(F.trim(F.col("indicador_obito").cast("string")) == "0", 0)
         .otherwise(F.lit(None).cast("int"))
    )
)

if sih_geo.filter(F.col("ano").isNull() | ~F.col("mes").between(1,12)).limit(1).count():
    raise RuntimeError("SIH possui competência inválida.")
validar_geografia(sih_geo, "SIH")
validar_fato_contra_dim_uf(sih_geo, "SIH", DIM_UF_PARA_VALIDACAO)

# Grão: UF de residência + município de residência + competência + CID principal + CNES.
# Uma linha NÃO equivale necessariamente a uma AIH; quantidade_aih é a medida somável.
gold_fato_internacoes = (
    sih_geo.groupBy(
        "uf_residencia", "codigo_uf_ibge", "municipio_codigo_6",
        "ano", "mes", "cid_codigo", "cnes_estabelecimento"
    ).agg(
        F.count("*").alias("quantidade_aih"),
        F.sum("valor_total").alias("valor_total"),
        F.sum("valor_uti").alias("valor_uti"),
        F.sum("dias_permanencia").alias("total_dias_permanencia"),
        F.sum("obito_hospitalar").alias("obitos_hospitalares"),
        F.count("obito_hospitalar").alias("registros_com_obito_informado"),
        F.count("valor_total").alias("registros_com_valor_informado")
    )
    .withColumn("ano_mes", F.col("ano") * 100 + F.col("mes"))
    .withColumn("doenca_cardiovascular", F.col("cid_codigo").rlike(r"^I[0-9]{2}"))
    .withColumn("fonte", F.lit("SIH"))
    .withColumn("granularidade", F.lit("COMPETENCIA_MENSAL"))
)

validar_integralidade("SIH", gold_fato_internacoes, "quantidade_aih", TOTAL_SIH)
print("TIPO codigo_uf_ibge:", gold_fato_internacoes.schema["codigo_uf_ibge"].dataType)
display(gold_fato_internacoes.limit(10))

## 07 — Fato SIM: óbitos agregados

In [ ]:
sim_geo = adicionar_geografia(sim, "municipio_residencia")
sim_geo = (
    sim_geo.withColumn("ano", F.year("data_obito"))
    .withColumn("mes", F.month("data_obito"))
    .withColumn("cid_codigo", cid_padronizado(F.col("cid_causa_basica")))
)

if sim_geo.filter(F.col("ano").isNull() | ~F.col("mes").between(1,12)).limit(1).count():
    raise RuntimeError("SIM possui data_obito inválida.")
validar_geografia(sim_geo, "SIM")
validar_fato_contra_dim_uf(sim_geo, "SIM", DIM_UF_PARA_VALIDACAO)

# Grão: UF de residência + município de residência + ano/mês de óbito + CID + sexo.
gold_fato_mortalidade = (
    sim_geo.groupBy(
        "uf_residencia", "codigo_uf_ibge", "municipio_codigo_6",
        "ano", "mes", "cid_codigo", "sexo_codigo"
    ).agg(F.count("*").alias("quantidade_obitos"))
    .withColumn("ano_mes", F.col("ano") * 100 + F.col("mes"))
    .withColumn("doenca_cardiovascular", F.col("cid_codigo").rlike(r"^I[0-9]{2}"))
    .withColumn("fonte", F.lit("SIM"))
    .withColumn("granularidade", F.lit("DATA_OBITO_AGREGADA_POR_MES"))
)

validar_integralidade("SIM", gold_fato_mortalidade, "quantidade_obitos", TOTAL_SIM)
print("TIPO codigo_uf_ibge:", gold_fato_mortalidade.schema["codigo_uf_ibge"].dataType)
display(gold_fato_mortalidade.limit(10))

## 08 — Fatos CNES, quando as Silver estiverem disponíveis

In [ ]:
TABELA_ST = "dbo.silver_cnes_estabelecimentos"
TABELA_LT = "dbo.silver_cnes_leitos"

TABELAS_GOLD = {
    "dbo.gold_dim_cid": gold_dim_cid,
    "dbo.gold_dim_tempo": gold_dim_tempo,
    "dbo.gold_fato_internacoes": gold_fato_internacoes,
    "dbo.gold_fato_mortalidade": gold_fato_mortalidade
}

if spark.catalog.tableExists(TABELA_ST) and spark.table(TABELA_ST).limit(1).count() > 0:
    st = spark.table(TABELA_ST)
    verificar_colunas(st, [
        "cnes_estabelecimento", "municipio_codigo", "ano_arquivo", "mes_arquivo"
    ], "CNES/ST")
    st_geo = adicionar_geografia(st, "municipio_codigo")
    validar_geografia(st_geo, "CNES/ST")
    validar_fato_contra_dim_uf(st_geo, "CNES/ST", DIM_UF_PARA_VALIDACAO)

    gold_fato_estabelecimentos = (
        st_geo.groupBy(
            "uf_residencia", "codigo_uf_ibge", "municipio_codigo_6",
            F.col("ano_arquivo").cast("int").alias("ano"),
            F.col("mes_arquivo").cast("int").alias("mes")
        )
        .agg(F.countDistinct("cnes_estabelecimento").alias("quantidade_estabelecimentos"))
        .withColumn("ano_mes", F.col("ano") * 100 + F.col("mes"))
        .withColumn("fonte", F.lit("CNES_ST"))
    )
    TABELAS_GOLD["dbo.gold_fato_estabelecimentos"] = gold_fato_estabelecimentos
else:
    print("CNES/ST não disponível na Silver: fato de estabelecimentos não será regravada.")

if spark.catalog.tableExists(TABELA_LT) and spark.table(TABELA_LT).limit(1).count() > 0:
    lt = spark.table(TABELA_LT)
    verificar_colunas(lt, [
        "cnes_estabelecimento", "municipio_codigo", "tipo_leito_codigo",
        "quantidade_existente", "quantidade_sus", "ano_arquivo", "mes_arquivo"
    ], "CNES/LT")
    lt_geo = adicionar_geografia(lt, "municipio_codigo")
    validar_geografia(lt_geo, "CNES/LT")
    validar_fato_contra_dim_uf(lt_geo, "CNES/LT", DIM_UF_PARA_VALIDACAO)

    # Quantidades são propriedades de tipos de leito; não são contagens de linhas LT.
    gold_fato_leitos = (
        lt_geo.groupBy(
            "uf_residencia", "codigo_uf_ibge", "municipio_codigo_6",
            F.col("ano_arquivo").cast("int").alias("ano"),
            F.col("mes_arquivo").cast("int").alias("mes"),
            "tipo_leito_codigo"
        )
        .agg(
            F.sum("quantidade_existente").alias("leitos_existentes"),
            F.sum("quantidade_sus").alias("leitos_sus")
        )
        .withColumn("ano_mes", F.col("ano") * 100 + F.col("mes"))
        .withColumn("fonte", F.lit("CNES_LT"))
    )
    TABELAS_GOLD["dbo.gold_fato_leitos"] = gold_fato_leitos
else:
    print("CNES/LT não disponível na Silver: fato de leitos não será regravada.")

print("Preparação CNES concluída.")

## 09 — Controle de cobertura, por fonte e frequência

In [ ]:
# SIH mensal por competência de arquivo/carga.
cobertura_sih = (
    sih.groupBy(
        F.col("uf_arquivo").alias("uf"),
        F.col("ano_competencia").cast("int").alias("ano"),
        F.col("mes_competencia").cast("int").alias("mes")
    )
    .agg(F.count("*").alias("registros"))
    .withColumn("fonte", F.lit("SIH"))
    .withColumn("abrangencia_arquivo", F.lit("ESTADUAL"))
    .withColumn("periodicidade", F.lit("MENSAL"))
)

# SIM: arquivo de origem ANUAL; manter mes=NULL aqui, ainda que a fato
# permita analisar a data do óbito mês a mês.
cobertura_sim = (
    sim.groupBy(F.col("uf_arquivo").alias("uf"), F.col("ano_arquivo").cast("int").alias("ano"))
    .agg(F.count("*").alias("registros"))
    .withColumn("mes", F.lit(None).cast("int"))
    .withColumn("fonte", F.lit("SIM"))
    .withColumn("abrangencia_arquivo", F.when(F.col("uf") == "BR", "NACIONAL").otherwise("ESTADUAL"))
    .withColumn("periodicidade", F.lit("ANUAL"))
)

coberturas = [cobertura_sih, cobertura_sim]
for fonte, tabela in [("CNES_ST", TABELA_ST), ("CNES_LT", TABELA_LT)]:
    if not spark.catalog.tableExists(tabela):
        continue
    df = spark.table(tabela)
    if df.limit(1).count() == 0:
        continue
    cobertura = (
        df.groupBy(
            F.col("uf_arquivo").alias("uf"),
            F.col("ano_arquivo").cast("int").alias("ano"),
            F.col("mes_arquivo").cast("int").alias("mes")
        )
        .agg(F.count("*").alias("registros"))
        .withColumn("fonte", F.lit(fonte))
        .withColumn("abrangencia_arquivo", F.lit("ESTADUAL"))
        .withColumn("periodicidade", F.lit("MENSAL"))
    )
    coberturas.append(cobertura)

gold_controle_cobertura = coberturas[0]
for parte in coberturas[1:]:
    gold_controle_cobertura = gold_controle_cobertura.unionByName(parte)
gold_controle_cobertura = gold_controle_cobertura.withColumn("gerado_em", F.current_timestamp())

TABELAS_GOLD["dbo.gold_controle_cobertura"] = gold_controle_cobertura
display(gold_controle_cobertura.orderBy("fonte", "uf", "ano", "mes"))

## 10 — Indicadores agregados por UF e ano

In [ ]:
indicadores_sim = (
    sim_geo.groupBy("uf_residencia", "ano")
    .agg(
        F.count("*").alias("total_obitos"),
        F.sum(F.when(F.col("cid_codigo").rlike(r"^I[0-9]{2}"), 1).otherwise(0))
         .alias("obitos_cardiovasculares"),
        F.countDistinct("mes").alias("meses_com_obitos")
    )
)

indicadores_sih = (
    sih_geo.groupBy("uf_residencia", "ano")
    .agg(
        F.count("*").alias("aih_competencias_carregadas"),
        F.sum(F.when(F.col("cid_codigo").rlike(r"^I[0-9]{2}"), 1).otherwise(0))
         .alias("aih_cardiovasculares_carregadas"),
        F.sum("valor_total").alias("valor_sih_competencias_carregadas"),
        F.countDistinct("mes").alias("meses_sih_carregados")
    )
)

gold_indicadores_uf_ano = (
    indicadores_sim.join(indicadores_sih, on=["uf_residencia", "ano"], how="full_outer")
    .withColumn("uf_nome", F.element_at(UF_NOME_EXPR, F.col("uf_residencia")))
    .withColumn("sih_ano_completo_carregado", F.coalesce(F.col("meses_sih_carregados") == 12, F.lit(False)))
    .withColumn("mortalidade_com_cobertura_temporal_verificada", F.lit(False))
)
# Importante: a ausência de SIH para determinada UF não é transformada em zero.
TABELAS_GOLD["dbo.gold_indicadores_uf_ano"] = gold_indicadores_uf_ano
display(gold_indicadores_uf_ano.orderBy("uf_residencia", "ano"))

## 11 — Validação GLOBAL antes de gravar

In [ ]:
print("=" * 70)
print("VALIDAÇÃO GLOBAL GOLD — PRÉ-GRAVAÇÃO")
print("=" * 70)

# Conferir novamente valores e tipos das duas fatos centrais.
for nome, df, medida, total in [
    ("SIH", gold_fato_internacoes, "quantidade_aih", TOTAL_SIH),
    ("SIM", gold_fato_mortalidade, "quantidade_obitos", TOTAL_SIM)
]:
    validar_integralidade(nome, df, medida, total)
    if not isinstance(df.schema["codigo_uf_ibge"].dataType, T.IntegerType):
        raise RuntimeError(f"{nome}: codigo_uf_ibge não está como INTEGER")
    if df.filter(F.col("codigo_uf_ibge").isNull()).limit(1).count():
        raise RuntimeError(f"{nome}: há código UF nulo")
    if df.filter(F.col("ano_mes").isNull()).limit(1).count():
        raise RuntimeError(f"{nome}: há competência ano_mes nula")
    print(nome, "chave UF INT e ano_mes válidos")

# Dimensões conformadas devem conter os códigos presentes nas fatos.
for nome, fato in [("SIH", gold_fato_internacoes), ("SIM", gold_fato_mortalidade)]:
    if (fato.select("cid_codigo").distinct()
        .join(gold_dim_cid.select("cid_codigo"), "cid_codigo", "left_anti")
        .limit(1).count()):
        raise RuntimeError(f"{nome}: CID sem dimensão")
    if (fato.select("ano_mes").distinct()
        .join(gold_dim_tempo.select("ano_mes"), "ano_mes", "left_anti")
        .limit(1).count()):
        raise RuntimeError(f"{nome}: ano_mes sem dimensão")

# Planilha de execução: informa tabelas que serão reconstruídas.
TABELAS_GERADAS = {}
for nome, df in TABELAS_GOLD.items():
    n = df.count()
    if n <= 0:
        raise RuntimeError(f"Tabela Gold vazia: {nome}")
    TABELAS_GERADAS[nome] = int(n)
    print(f"PRONTA: {nome} | {n} linhas")

print("\nTodos os testes pré-gravação foram concluídos.")
if not APLICAR_GRAVACAO:
    print("MODO SEGURO: não há escrita. Revise os números e faça backup antes de ativar.")

## 12 — Persistir Gold / Delta Lake (somente com autorização explícita)

In [ ]:
if not APLICAR_GRAVACAO:
    print("GRAVAÇÃO NÃO EXECUTADA (APLICAR_GRAVACAO=False).")
else:
    if not CONFIRMO_LAKEHOUSE_ORIGINAL:
        raise RuntimeError("Lakehouse de origem não confirmado.")

    # A substituição de múltiplas tabelas não é atômica.
    # Criar backup/versionamento externo antes de regravações produtivas.
    for nome, df in TABELAS_GOLD.items():
        print("GRAVANDO:", nome)
        (
            df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(nome)
        )
        atual = spark.table(nome)
        total = atual.count()
        if total != TABELAS_GERADAS[nome]:
            raise RuntimeError(f"Divergência após gravação de {nome}: {total}")
        print("OK:", nome, "|", total, "linhas")

    if not isinstance(spark.table("dbo.gold_fato_internacoes")
                      .schema["codigo_uf_ibge"].dataType, T.IntegerType):
        raise RuntimeError("Chave UF persistida não está em INTEGER!")

    print("TODAS AS TABELAS GOLD SELECIONADAS FORAM GRAVADAS.")

## 13 — Auditoria e conclusão

In [ ]:
FIM = datetime.now(timezone.utc).replace(tzinfo=None)
TABELA_CONTROLE = "dbo.media_controle_execucoes"

if not APLICAR_GRAVACAO:
    print("EXECUÇÃO DE VALIDAÇÃO: nenhuma tabela Gold foi sobrescrita.")
    print("Tabelas inspecionadas:", len(TABELAS_GERADAS))
else:
    if not spark.catalog.tableExists(TABELA_CONTROLE):
        raise RuntimeError("Tabela de auditoria ausente. Execute nb_media_00_setup.")

    schema_log = T.StructType([
        T.StructField("execution_id", T.StringType()),
        T.StructField("notebook", T.StringType()),
        T.StructField("fonte", T.StringType()),
        T.StructField("camada", T.StringType()),
        T.StructField("uf", T.StringType()),
        T.StructField("ano", T.IntegerType()),
        T.StructField("mes", T.IntegerType()),
        T.StructField("arquivo", T.StringType()),
        T.StructField("status", T.StringType()),
        T.StructField("registros", T.LongType()),
        T.StructField("mensagem", T.StringType()),
        T.StructField("iniciado_em", T.TimestampType()),
        T.StructField("finalizado_em", T.TimestampType())
    ])

    linhas = [(
        EXECUTION_ID, NOTEBOOK, "MULTIFONTE", "GOLD", "BR", None, None,
        tabela, "SUCESSO", int(quantidade),
        "Tabela Gold reconstruída; codigo_uf_ibge convertido para INTEGER nas fatos.",
        INICIO, FIM
    ) for tabela, quantidade in TABELAS_GERADAS.items()]

    df_log = spark.createDataFrame(linhas, schema_log)
    df_log.write.format("delta").mode("append").saveAsTable(TABELA_CONTROLE)
    print("Auditoria registrada em", TABELA_CONTROLE)

print("=" * 70)
print("MedIA | Notebook 06 concluído", "| Execução:", EXECUTION_ID)
print("=" * 70)

print("\nPRÓXIMO PASSO NO POWER BI:")
print("1. Atualizar o esquema do modelo semântico após a regravação.")
print("2. gold_dim_uf[uf_codigo] (1) -> (*) gold_fato_internacoes[codigo_uf_ibge].")
print("3. Usar filtro único, ativo, após conferir integridade.")
print("4. Repetir UF -> mortalidade e CID -> fatos.")
print("5. Usar SUM(quantidade_aih) e SUM(quantidade_obitos) como medidas, NÃO COUNTROWS.")